In [25]:
# ── Cell 1: Imports ────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [27]:
# ── Cell 2: Paths ──────────────────────────────────────────────
DATA_DIR  = '/kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign'
TRAIN_DIR = os.path.join(DATA_DIR, 'Train')
TEST_DIR  = os.path.join(DATA_DIR, 'Test')

In [28]:
# ── Cell 3: Constants ──────────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_CLASSES = 43
EPOCHS      = 20
SEED        = 42

In [29]:
# ── Cell 4: Load CSV + fix paths + split ───────────────────────
train_df = pd.read_csv(os.path.join(DATA_DIR, 'Train.csv'))

train_df['full_path'] = train_df['Path'].apply(
    lambda x: os.path.join(DATA_DIR, x)
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_df['ClassId']
)

print(f"Train samples: {len(train_df)}")
print(f"Val samples:   {len(val_df)}")

Train samples: 31367
Val samples:   7842


In [31]:
# ── Cell 5: tf.data pipeline (EfficientNet version) ────────────
def parse_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    # EfficientNet expects [0, 255] — no normalization needed
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

def augment(img, label):
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.rot90(img, k=tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32))
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label

def make_dataset(df, augment_data=False):
    paths  = df['full_path'].values
    labels = df['ClassId'].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

In [32]:
# ── Cell 6: Create datasets ────────────────────────────────────
train_ds = make_dataset(train_df, augment_data=True)
val_ds   = make_dataset(val_df,   augment_data=False)

print(f"Train batches: {len(train_ds)}")
print(f"Val batches:   {len(val_ds)}")

Train batches: 981
Val batches:   246


In [33]:
# ── Cell 7: Verify one batch ───────────────────────────────────
for images, labels in train_ds.take(1):
    print(f"Batch image shape: {images.shape}")
    print(f"Batch label shape: {labels.shape}")
    print(f"Pixel range: [{images.numpy().min():.2f}, {images.numpy().max():.2f}]")

Batch image shape: (32, 224, 224, 3)
Batch label shape: (32, 43)
Pixel range: [0.00, 255.00]


In [ ]:
# ── Cell 8: Build EfficientNetB0 — Feature Extraction ─────────
from tensorflow.keras.applications import EfficientNetB0
import time

def build_efficientnet(num_classes=43):
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = False

    inputs = Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model, base_model

effnet_model, effnet_base = build_efficientnet()
effnet_model.summary()

In [ ]:
# ── Cell 9: Train EfficientNet — Feature Extraction ───────────
effnet_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_eff = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('efficientnet_best.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

start = time.time()
history_effnet = effnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks_eff
)
effnet_train_time = time.time() - start
print(f"Training time: {effnet_train_time/60:.1f} minutes")

plot_history(history_effnet, title='EfficientNetB0 — Feature Extraction')

In [ ]:
# ── Save after feature extraction ──────────────────────────────
effnet_model.save('/kaggle/working/efficientnet_feature_extraction.keras')
print("Feature extraction model saved.")

In [ ]:
# ── Plot training history ───────────────────────────────────────
def plot_history(history, title='Model'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_title(f'{title} — Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    axes[1].plot(history.history['accuracy'], label='Train Acc')
    axes[1].plot(history.history['val_accuracy'], label='Val Acc')
    axes[1].set_title(f'{title} — Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_history(history_effnet, title='EfficientNetB0 — Feature Extraction')

In [ ]:
# ── Cell 10: Fine-tune EfficientNetB0 ──────────────────────────
# EfficientNetB0 has 237 layers — unfreeze last 20
effnet_base.trainable = True

for layer in effnet_base.layers[:-20]:
    layer.trainable = False

for layer in effnet_base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable = sum(1 for l in effnet_base.layers if l.trainable)
frozen = sum(1 for l in effnet_base.layers if not l.trainable)
print(f"Trainable layers: {trainable}")
print(f"Frozen layers:    {frozen}")

effnet_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_eff_ft = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('efficientnet_finetuned.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

start = time.time()
history_effnet_ft = effnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks_eff_ft
)
effnet_ft_time = time.time() - start
print(f"Fine-tuning time: {effnet_ft_time/60:.1f} minutes")

plot_history(history_effnet_ft, title='EfficientNetB0 — Fine-Tuning')

In [ ]:


# ── Save after fine-tuning ──────────────────────────────────────
effnet_model.save('/kaggle/working/efficientnet_finetuned_final.keras')
print("Fine-tuned model saved.")

In [ ]:
# ── Create test dataset ─────────────────────────────────────────
test_df = pd.read_csv(os.path.join(DATA_DIR, 'Test.csv'))
test_df['full_path'] = test_df['Path'].apply(lambda x: os.path.join(DATA_DIR, x))
test_ds = make_dataset(test_df, augment_data=False)
print(f"Test batches: {len(test_ds)}")

In [ ]:
# ── Cell 11: Evaluate EfficientNet + comparison ────────────────
import time

# Evaluate EfficientNet on test set
loss_eff, acc_eff = effnet_model.evaluate(test_ds, verbose=1)
print(f"EfficientNet Test Accuracy: {acc_eff*100:.2f}%")

# EfficientNet inference speed
def measure_inference(model, dataset, n_batches=10):
    start = time.time()
    for i, (imgs, _) in enumerate(dataset):
        if i >= n_batches:
            break
        model.predict(imgs, verbose=0)
    return (time.time() - start) / n_batches

effnet_inference = measure_inference(effnet_model, test_ds)

# Feature extraction val accuracy (last epoch)
eff_feat_val = history_effnet.history['val_accuracy'][-1] * 100

# Comparison table — ResNet numbers from previous notebook
print("\n" + "="*62)
print(f"{'Metric':<32} {'ResNet50':>12} {'EfficientNetB0':>15}")
print("="*62)
print(f"{'Parameters':<32} {'25.6M':>12} {'5.3M':>15}")
print(f"{'Val Acc (feature extraction)':<32} {'96.7%':>12} {f'{eff_feat_val:.1f}%':>15}")
print(f"{'Val Acc (fine-tuned)':<32} {'99.6%':>12} {f'{acc_eff*100:.1f}%':>15}")
print(f"{'Test Accuracy':<32} {'88.9%':>12} {f'{acc_eff*100:.1f}%':>15}")
print(f"{'Train time (feature extract)':<32} {'35 min':>12} {'16.9 min':>15}")
print(f"{'Train time (fine-tuning)':<32} {'39 min':>12} {'16.4 min':>15}")
print(f"{'Inference time/batch':<32} {'~0.09s':>12} {f'{effnet_inference:.3f}s':>15}")
print("="*62)

In [34]:
#load model
effnet_model = tf.keras.models.load_model('/kaggle/working/efficientnet_finetuned_final.keras')


In [ ]:
#  Get predictions on test set ───────────────────────
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = effnet_model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(f"Total test samples: {len(y_true)}")
print(f"Overall accuracy: {(y_true == y_pred).mean()*100:.2f}%")

In [ ]:
#  Confusion Matrix ───────────────────────────────────
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(20, 16))
sns.heatmap(
    cm,
    annot=False,
    fmt='d',
    cmap='Blues',
    xticklabels=range(43),
    yticklabels=range(43)
)
plt.title('Confusion Matrix — EfficientNetB0', fontsize=14)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
#  Classification Report ─────────────────────────────
from sklearn.metrics import classification_report

report = classification_report(y_true, y_pred, digits=3)
print(report)

In [ ]:
#  Per-class accuracy + worst classes ─────────────────
per_class_acc = cm.diagonal() / cm.sum(axis=1)

sorted_idx = np.argsort(per_class_acc)

print("10 hardest classes to classify:")
print(f"{'Class':<10} {'Accuracy':<12} {'Total Samples'}")
print("-" * 35)
for idx in sorted_idx[:10]:
    total = cm.sum(axis=1)[idx]
    print(f"Class {idx:<5} {per_class_acc[idx]*100:<12.1f}% {total}")

print("\n10 easiest classes to classify:")
print(f"{'Class':<10} {'Accuracy':<12} {'Total Samples'}")
print("-" * 35)
for idx in sorted_idx[-10:]:
    total = cm.sum(axis=1)[idx]
    print(f"Class {idx:<5} {per_class_acc[idx]*100:<12.1f}% {total}")

In [ ]:
# ──  Visualize worst predicted classes ──────────────────
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for plot_idx, class_idx in enumerate(sorted_idx[:10]):
    mask = (y_true == class_idx) & (y_pred != class_idx)
    wrong_indices = np.where(mask)[0]

    ax = axes[plot_idx]
    if len(wrong_indices) > 0:
        wrong_preds = y_pred[wrong_indices]
        most_common_wrong = np.bincount(wrong_preds).argmax()
        ax.set_title(
            f'True: {class_idx}\nPredicted as: {most_common_wrong}\nAcc: {per_class_acc[class_idx]*100:.0f}%',
            fontsize=9
        )
    else:
        ax.set_title(f'Class {class_idx}', fontsize=9)
    ax.axis('off')

plt.suptitle('10 Most Misclassified Classes — EfficientNetB0', fontsize=13)
plt.tight_layout()
plt.show()